# 체인(Chain)이란?

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. 체이닝(Chaining) 개념을 이해하고 `PromptTemplate | Model | OutputParser` 구조를 설명할 수 있어요
2. 파이프(`|`) 연산자로 체인을 구성하고 `invoke()`로 실행할 수 있어요
3. `RunnableLambda`로 일반 Python 함수를 체인에 삽입할 수 있어요
4. 여러 LLM 호출을 연결하는 다단계 체인을 만들 수 있어요

## 사전 지식

- `01_LangChain Quick Start.ipynb`의 `invoke()`, `AIMessage` 개념
- Python 함수, 람다 표현식 기초

## 이전 노트북 연결

`01_LangChain Quick Start.ipynb`에서 모델에 직접 문자열이나 메시지 객체를 전달했어요. 이제 프롬프트 템플릿(Prompt Template)과 출력 파서(Output Parser)를 파이프 연산자로 연결해 재사용 가능한 체인을 만들어볼게요.

아래 다이어그램은 이 노트북에서 다룰 체인 흐름을 보여줘요.

```mermaid
flowchart LR
    A["딕셔너리 입력<br/>{topic: ...}"]:::input --> B["PromptTemplate<br/>템플릿 채우기"]:::process
    B --> C["ChatOpenAI<br/>LLM 호출"]:::process
    C --> D["StrOutputParser<br/>텍스트 추출"]:::process
    D --> E["RunnableLambda<br/>커스텀 변환"]:::process
    E --> F["최종 출력<br/>문자열 또는 딕셔너리"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

체이닝은 여러 작업을 연속으로 연결해 실행하는 방식이에요. 한 작업의 결과가 다음 작업의 입력으로 이어지므로, 복잡한 워크플로우(Workflow)를 간결하게 표현할 수 있어요.

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

load_dotenv()

# 모델 초기화
# temperature 미지정 시 기본값 사용 (적당한 창의성)
model = ChatOpenAI(model="gpt-4o-mini")

## 1. 기본적인 체이닝: PromptTemplate + OutputParser

`PromptTemplate`과 `StrOutputParser(String Output Parser)`를 파이프(`|`) 연산자로 연결해 첫 번째 체인을 만들어볼게요.

- `PromptTemplate.from_template()`: `{변수명}` 형식의 플레이스홀더를 자동으로 인식해요
- `StrOutputParser`: `AIMessage`에서 `.content` 문자열만 추출해요
- 파이프 연산자: 왼쪽 출력이 오른쪽 입력으로 전달돼요

In [ ]:
# ---------------------------------------------------
# PromptTemplate + Model + OutputParser 체인 구성하기
# ---------------------------------------------------
# ============================================================
# TODO: {question} 자리에 원하는 질문이 들어가도록 invoke()를 호출해요
# 힌트: invoke()에 딕셔너리 {"question": "..."} 형태로 전달해요
# 예상 결과: 질문에 대한 LLM 응답 문자열이 출력돼요
# ============================================================

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# 1. 변수 자동 인식
prompt_template = PromptTemplate.from_template(
  """
  다음 질문에 친절하게 대답해
  {question}
  """
)
prompt_template
print(f' ==> [Line 13]: \033[38;2;208;173;11m[prompt_template]\033[0m({type(prompt_template).__name__}) = \033[38;2;96;163;250m{prompt_template}\033[0m')

# 2. StrOutputParser 생성
# StrOutputParser : AIMessage에서 .content 문자열만 추출

output_parser = StrOutputParser()

# 3. 체인 생성
chain = prompt_template | model | output_parser
chain

# 4. 체인 실행
result = chain.invoke({"question": "점심추천해줘!"})
print(f' ==> [Line 32]: \033[38;2;3;191;250m[result]\033[0m({type(result).__name__}) = \033[38;2;221;104;121m{result}\033[0m')



 ==> [Line 13]: [prompt_template](PromptTemplate) = input_variables=['question'] input_types={} partial_variables={} template='\n  다음 질문에 친절하게 대답해\n  {question}\n  '
 ==> [Line 32]: [result](AIMessage) = content='점심으로는 어떤 음식을 좋아하시는지에 따라 다양한 추천이 가능합니다! 다음과 같은 옵션을 고려해보세요:\n\n1. **한식**: 비빔밥이나 불고기 덮밥, 김치찌개와 밥은 항상 좋은 선택이에요.\n2. **중식**: 짜장면이나 탕수육, 팔보채 같은 메뉴도 맛있습니다.\n3. **일식**: 초밥이나 덮밥(돈부리), 라멘 같은 것도 좋고요.\n4. **양식**: 파니니나 샌드위치, 샐러드와 함께 수프 한 그릇도 괜찮아요.\n5. **간편식**: 건강한 샐러드나 샌드위치, 또는 베이글도 좋은 선택이 될 수 있어요.\n\n원하는 음식 스타일이나 알레르기, 특별한 다이어트 효과가 필요하시면 말씀해 주세요!' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 211, 'prompt_tokens': 27, 'total_tokens': 238, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_668125ce6a', 'id': 'chatcmpl-DMPIJsAye

## 2. 여러 단계를 거치는 체이닝 (PromptTemplate + RunnableLambda)

PromptTemplate과 RunnableLambda를 사용하여 여러 단계를 파이프로 연결합니다.

> **한계점**: 아래 예제에서는 `add_topic_info` 함수 내에 `"파이썬"`이 하드코딩되어 있습니다. 파이프라인 중간에서 원본 입력값을 유지하려면 `RunnablePassthrough.assign()`이나 `RunnableParallel`을 사용해야 합니다. 이 패턴은 `04_Runnable_Basic.ipynb`에서 자세히 다룹니다.

In [12]:
# ---------------------------------------------------
# RunnableLambda로 중간 변환 함수 삽입하기
# ---------------------------------------------------

prompt_template = PromptTemplate.from_template(
  "{topic}에 대해 간단히 설명해줘."
)

# 사용자 함수
def format_output(data):
  topic = data["topic"]
  response_text = data["response"]

  return f"질문: {topic}\n 답변: {response_text}"

# 유저의 입력 토픽을 유지하면서 체인 구성
def create_formatted_chain():
  def add_topic_info(response_text):
    return {"topic": "파이썬", "response": response_text}
  
  return (
    prompt_template | model | StrOutputParser() | RunnableLambda(add_topic_info) | RunnableLambda(format_output)
  )

chain = create_formatted_chain()
print(chain.invoke({"topic": "파이썬"}))

# data = {
#   "topic": "",
#   "response": ""
# }
# format_output(data)

질문: 파이썬
 답변: 파이썬(Python)은 1991년 귀도 반 로썸(Guido van Rossum)에 의해 처음 개발된 고급 프로그래밍 언어입니다. 파이썬은 문법이 간결하고 가독성이 높아 초보자와 전문가 모두에게 인기 있는 언어입니다. 다음은 파이썬의 주요 특징입니다.

1. **간결한 문법**: 파이썬은 코드가 쉽게 읽히고 이해하기 쉽도록 설계되었습니다. 들여쓰기를 사용하여 블록을 구분하므로 시각적으로 깔끔합니다.

2. **다양한 용도**: 웹 개발, 데이터 분석, 인공지능, 머신러닝, 과학 계산 등 여러 분야에서 널리 사용됩니다.

3. **풍부한 라이브러리**: 데이터 과학 라이브러리인 NumPy, Pandas, 머신러닝 라이브러리인 TensorFlow, Scikit-learn, 웹 프레임워크인 Django, Flask 등 다양한 라이브러리가 있습니다.

4. **플랫폼 독립성**: Windows, macOS, Linux 등 다양한 운영체제에서 동작합니다.

5. **인터프리터 언어**: 코드를 작성한 후 즉시 실행할 수 있어 개발 속도가 빠릅니다.

파이썬은 강력하면서도 배우기가 쉬워서, 프로그래밍을 처음 시작하는 사람들에게 매우 적합한 언어입니다.


## 3. Runnable 체인을 함수로 만들기

앞서 `RunnableLambda`로 중간 변환을 삽입했어요. 체인 생성 로직 자체를 함수로 캡슐화하면 여러 곳에서 재사용하기가 편해요.

다음 셀에서 체인 생성 함수를 만들고 호출해볼게요.

In [13]:
# ---------------------------------------------------
# 체인 생성 로직을 함수로 캡슐화하기
# ---------------------------------------------------
# ============================================================
# TODO: create_explanation_chain()을 호출하고 다른 topic으로 실행해봐요
# 힌트: chain.invoke({"topic": "원하는 주제"}) 형태로 호출해요
# 예상 결과: 입력 주제에 대한 설명 문자열이 출력돼요
# ============================================================


def create_explaination_chain():
  prompt_template = PromptTemplate.from_template(
    "{topic}에 대해 간단히 설명해줘"
  )
  return prompt_template | model | StrOutputParser()

chain = create_explaination_chain()
result = chain.invoke({"topic": "머신러닝"})
print(f' ==> [Line 18]: \033[38;2;190;217;221m[result]\033[0m({type(result).__name__}) = \033[38;2;222;204;30m{result}\033[0m')


 ==> [Line 18]: [result](str) = 머신러닝(Machine Learning)은 인공지능(AI)의 한 분야로, 컴퓨터가 명시적으로 프로그램되지 않고도 주어진 데이터에서 패턴을 학습하고 예측을 수행할 수 있도록 하는 기술입니다. 머신러닝의 핵심은 알고리즘을 통해 데이터를 분석하고, 그 결과를 바탕으로 모델을 만들어 새로운 데이터에 대한 예측이나 결정을 내리는 것입니다.

머신러닝은 주로 세 가지 유형으로 나눌 수 있습니다:

1. **지도 학습(Supervised Learning)**: 레이블이 있는 데이터를 사용하여 모델을 학습시킵니다. 입력과 출력 간의 관계를 학습하여 주어진 새로운 입력에 대해 예측을 수행합니다. 예를 들어, 스팸 이메일 필터링이 있습니다.

2. **비지도 학습(Unsupervised Learning)**: 레이블이 없는 데이터를 사용하여 데이터의 숨겨진 구조나 패턴을 찾습니다. 클러스터링이나 데이터 차원 축소가 이에 해당합니다. 예를 들어, 고객 세분화가 있습니다.

3. **강화 학습(Reinforcement Learning)**: 에이전트가 환경과 상호작용하면서 보상을 최대화하기 위해 학습하는 방법입니다. 게임이나 로봇 제어와 같은 분야에서 많이 사용됩니다.

머신러닝은 이미지 인식, 자연어 처리, 추천 시스템 등 다양한 분야에서 활용되고 있습니다.


## 4. 파이프 연산자(`|`)를 사용한 체이닝

재사용 가능한 체인 함수를 만들어봤어요. 이제 파이프 연산자가 기존 방식과 어떻게 다른지 직접 비교해볼게요.

파이프 연산자를 사용하면 중간 변수 없이 `PromptTemplate | model | StrOutputParser()` 형태로 한 줄에 표현할 수 있어요. 데이터가 왼쪽에서 오른쪽으로 흐르는 방향이 코드에서 바로 보여요.

In [23]:
# ---------------------------------------------------
# 파이프 없이 각 단계를 수동으로 실행하는 방식 (비교용)
# ---------------------------------------------------
# 각 단계의 출력을 중간 변수에 담아 다음 단계로 전달해야 해요
prompt_template = PromptTemplate.from_template("사용자의 질문에 대해 친절하게 답변해 : {question}")
prompt_value = prompt_template.invoke({"question": "파이썬에 대해 간단히 알려줘"})
print(f' ==> [Line 6]: \033[38;2;236;109;74m[prompt_value]\033[0m({type(prompt_value).__name__}) = \033[38;2;11;53;252m{prompt_value}\033[0m')

model_response = model.invoke(prompt_value)
print(f' ==> [Line 9]: \033[38;2;207;226;66m[model_response]\033[0m({type(model_response).__name__}) = \033[38;2;14;103;249m{model_response}\033[0m')

output_parser = StrOutputParser()
output_parser_res = output_parser.invoke(model_response)
print(f' ==> [Line 13]: \033[38;2;41;106;64m[output_parser_res]\033[0m({type(output_parser_res).__name__}) = \033[38;2;136;11;156m{output_parser_res}\033[0m')


 ==> [Line 6]: [prompt_value](StringPromptValue) = text='사용자의 질문에 대해 친절하게 답변해 : 파이썬에 대해 간단히 알려줘'
 ==> [Line 9]: [model_response](AIMessage) = content='파이썬(Python)은 강력하고 유연한 프로그래밍 언어로, 1991년에 귀도 반 로섬(Guido van Rossum)에 의해 처음 발표되었습니다. 다음은 파이썬의 몇 가지 주요 특징입니다.\n\n1. **간결하고 읽기 쉬운 문법**: 파이썬은 코드가 매우 직관적이며, 가독성이 높아 초보자도 쉽게 배울 수 있습니다.\n\n2. **다양한 용도**: 웹 개발, 데이터 분석, 인공지능, 자동화 스크립팅, 게임 개발 등 여러 분야에서 사용됩니다.\n\n3. **광범위한 라이브러리**: 수많은 외부 라이브러리와 프레임워크(예: Django, Flask, Pandas, NumPy 등)를 통해 다양한 기능을 쉽게 구현할 수 있습니다.\n\n4. **크로스 플랫폼**: 파이썬은 Windows, macOS, Linux 등 다양한 운영체제에서 실행될 수 있습니다.\n\n5. **대화형 인터프리터**: 파이썬은 대화형 인터프리터를 제공하여, 코드를 한 줄씩 입력하고 바로 실행 결과를 확인할 수 있어 학습과 테스트에 유용합니다.\n\n파이썬은 그 유연성과 커뮤니티의 지원 덕분에 현재 가장 인기 있는 프로그래밍 언어 중 하나로 자리 잡고 있습니다. 파이썬을 배우면 여러 분야에서 활용할 수 있는 유용한 기술을 익힐 수 있습니다! 추가적인 질문이 있으면 언제든지 물어주세요.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 340, 'prompt_tokens': 30, 'total_tokens': 370, 'completion_tokens_details': {'accepted_prediction_tokens

파이프 연산자를 사용하면 위의 코드를 더 간결하게 작성할 수 있습니다:


In [ ]:
# ---------------------------------------------------
# 파이프 연산자로 체인을 한 줄에 표현하기
# ---------------------------------------------------
# ============================================================
# TODO: summary_prompt 변수를 정의하고 파이프로 chain을 구성해요
# 힌트: PromptTemplate.from_template("{text}") | model | StrOutputParser()
# 예상 결과: 입력 텍스트를 한 문장으로 요약한 결과가 출력돼요
# ============================================================

summary_prompt = PromptTemplate.from_template(
  """다음 문장을 한 문장으로 요약해줘:
  {text}
  """
)
text = """파이썬은 배우기 쉽고 강력한 프로그래밍 언어로 웹개발 등등 다양한 분야에서 사용됩니다."""
chain = summary_prompt | model | StrOutputParser()
print(chain.invoke({"text": text}))

### 파이프 연산자의 동작 원리

파이프 연산자(`|`)는 왼쪽의 Runnable 결과를 오른쪽 Runnable의 입력으로 전달합니다:
- `PromptTemplate | model` → 프롬프트 결과를 model에 전달
- `model | OutputParser` → model의 결과를 OutputParser에 전달
- `PromptTemplate | model | OutputParser` → 순차적으로 연결

데이터가 왼쪽에서 오른쪽으로 흐르는 것을 시각적으로 표현할 수 있어 코드가 더 읽기 쉬워집니다.

> **참고**: LangChain에서는 `__or__` 메서드를 오버라이드하여 `|` 연산자로 체인을 연결할 수 있어요. 이는 특정 Python 버전에 의존하지 않으며, LangChain이 지원하는 모든 Python 버전에서 동작합니다.


In [ ]:
# ---------------------------------------------------
# RunnableLambda로 후처리 함수를 체인에 추가하기
# ---------------------------------------------------
prompt_template = PromptTemplate.from_template("{question}")

def format_output(res_text):
  return f"⛓️답변⛓️ \n {res_text}"

chain = (
  prompt_template
  | model
  | StrOutputParser()
  | RunnableLambda(format_output)
)

print(chain.invoke("머신러닝이란 무엇인가요?"))



⛓️답변⛓️ 
 머신러닝(머신 러닝, Machine Learning)은 인공지능(AI)의 한 분야로, 컴퓨터가 데이터를 통해 학습하고 경험을 바탕으로 자동으로 개선하는 기술입니다. 즉, 머신러닝은 명시적으로 프로그래밍되지 않더라도, 데이터를 통해 패턴을 인식하고 예측을 할 수 있도록 하는 알고리즘과 모델을 개발하는 과정입니다.

머신러닝은 크게 세 가지 유형으로 나눌 수 있습니다:

1. **지도 학습(Supervised Learning)**: 입력 데이터와 해당 입력에 대한 정답(레이블)이 주어지고, 모델이 이 데이터를 학습하여 새로운 데이터에 대한 예측을 수행하는 방식입니다. 예를 들어, 스팸 메일 필터링, 이미지 인식 등이 있습니다.

2. **비지도 학습(Unsupervised Learning)**: 입력 데이터에 레이블이 없는 경우, 데이터의 구조나 패턴을 발견하는 데 초점을 맞춥니다. 클러스터링, 차원 축소, 연관 규칙 학습 등이 대표적인 예입니다.

3. **강화 학습(Reinforcement Learning)**: 에이전트가 환경과 상호작용하며 보상을 최대화하기 위해 학습하는 방식입니다. 게임 플레이, 로봇 공학 등이 이 범주에 포함됩니다.

머신러닝은 데이터 분석, 자연어 처리, 이미지 처리 등 다양한 분야에서 활용되며, 최근에는 딥러닝과 같은 더 발전된 기술이 많은 주목을 받고 있습니다.


### 파이프 연산자의 장점

- **가독성**: 코드의 흐름이 왼쪽에서 오른쪽으로 자연스럽게 읽힙니다
  - `입력 → 처리1 → 처리2 → 출력` 형태로 데이터 흐름을 직관적으로 표현
- **간결성**: 중간 변수를 줄여 코드가 더 짧아집니다
- **연결성**: 여러 단계를 쉽게 연결할 수 있습니다
  - 새로운 단계를 추가할 때 `| 새로운단계`만 붙이면 됩니다


In [ ]:
# ---------------------------------------------------
# 파이프 사용 전·후 코드 비교하기
# ---------------------------------------------------


## 5. 여러 LLM 호출을 연결하는 체인

파이프 연산자의 장점을 확인했어요. 이제 한 LLM의 출력을 다음 LLM의 입력으로 연결하는 다단계 체인을 만들어볼게요.

첫 번째 체인이 설명을 생성하고, 두 번째 체인이 그 설명을 요약해요. 두 체인을 하나로 합칠 때 `RunnableLambda`로 데이터 형식(문자열 → 딕셔너리)을 변환하는 방법도 확인할 수 있어요.

In [26]:
# ---------------------------------------------------
# 두 LLM 호출을 연결하는 다단계 체인 만들기
# ---------------------------------------------------
# ============================================================
# TODO: explanation_chain의 출력을 summary_chain의 입력으로 연결해요
# 힌트: RunnableLambda(lambda text: {"text": text})로 문자열→딕셔너리 변환
# 예상 결과: "파이썬"에 대한 설명을 생성하고, 그것을 다시 한 문장으로 요약해요
# ============================================================

# 1단계: 설명 생성 체인
explanation_prompt = PromptTemplate.from_template("{topic}에 대해 간단히 설명해줘")
explanation_chain = explanation_prompt | model | StrOutputParser()

# 2단계: 요약 생성 체인
# StrOutputParser 출력(문자열)을 {text}에 넣어 요약 요청
summary_prompt = PromptTemplate.from_template(
    "다음 내용을 한 문장으로 요약해줘:\n\n{text}"
)
summary_chain = summary_prompt | model | StrOutputParser()

# 수동 연결 — 첫 번째 체인의 결과를 직접 두 번째 체인에 전달
explanation = explanation_chain.invoke({"topic": "파이썬"})
print("첫 번째 응답:")
print(explanation)
print("\n" + "="*50 + "\n")

summary = summary_chain.invoke({"text": explanation})
print("두 번째 응답 (요약):")
print(summary)

# 자동 연결 — RunnableLambda로 체인 간 데이터 형식을 변환
# 핵심: explanation_chain은 문자열 반환, summary_chain은 {"text": ...} 딕셔너리를 기대
print("\n" + "="*50)
print("한 번에 실행 (RunnableLambda 사용):")
full_chain = (
    explanation_chain 
    | RunnableLambda(lambda text: {"text": text})  # 문자열 → 딕셔너리 변환
    | summary_chain
)
final_result = full_chain.invoke({"topic": "파이썬"})
print(final_result)

첫 번째 응답:
파이썬(Python)은 1991년 귀도 반 로섬(Guido van Rossum)에 의해 개발된 고급 프로그래밍 언어입니다. 이 언어는 코드가 간결하고 읽기 쉬운 것이 특징으로, 문법이 명확하고 직관적이어서 초보자가 배우기에도 적합합니다.

파이썬의 주요 특징은 다음과 같습니다:

1. **읽기 쉬운 문법**: 파이썬은 코드가 명확하게 읽히도록 설계되어 있어, 다른 프로그래밍 언어에 비해 배우기 쉽습니다.

2. **다양한 용도**: 웹 개발, 데이터 분석, 인공지능, 머신러닝, 자동화 스크립트 등 다양한 분야에서 사용됩니다.

3. **풍부한 라이브러리**: 데이터 처리를 위한 NumPy, 데이터 분석을 위한 Pandas, 웹 프레임워크인 Django와 Flask 등 다양한 라이브러리와 프레임워크가 있습니다.

4. **인터프리터 언어**: 파이썬은 인터프리터 언어로, 작성한 코드를 즉시 실행할 수 있어 빠른 개발과 테스팅이 가능합니다.

5. **플랫폼 독립성**: 파이썬은 다양한 운영 체제에서 실행될 수 있어, 크로스 플랫폼 개발에 유리합니다.

이러한 특징 덕분에 파이썬은 많은 개발자와 기업에서 선호하는 언어로 자리 잡고 있습니다.


두 번째 응답 (요약):
파이썬은 1991년 개발된 고급 프로그래밍 언어로, 읽기 쉬운 문법과 다양한 용도, 풍부한 라이브러리, 인터프리터 언어의 특징 등으로 인해 초보자와 개발자에게 선호받으며 크로스 플랫폼 개발에 적합합니다.

한 번에 실행 (RunnableLambda 사용):
파이썬은 귀도 반 로섬이 1991년에 개발한 고급 프로그래밍 언어로, 읽기 쉬운 문법과 다양한 용도, 광범위한 라이브러리, 플랫폼 독립성, 강력한 커뮤니티 덕분에 초보자부터 전문가까지 널리 사용되며, 특히 데이터 과학과 인공지능 분야에서 인기를 끌고 있습니다.


In [32]:
# ---------------------------------------------------
# RunnableLambda로 요약 프롬프트를 문자열로 직접 생성하기
# ---------------------------------------------------

# 첫 번째 단계: 설명 생성
explanation_prompt = PromptTemplate.from_template("{topic}에 대해 간단히 설명해줘")

# 두 번째 단계: 요약 프롬프트를 문자열로 직접 만들기
# model은 딕셔너리뿐 아니라 문자열 입력도 받아요 → PromptTemplate 없이 연결 가능
def create_summary_prompt(explanation_text):
    """설명 텍스트를 받아 요약 프롬프트 문자열로 변환"""
    return f"다음 내용을 한 문장으로 요약해줘:\n\n{explanation_text}"

explanation_chain = (
    explanation_prompt
    | model 
    | StrOutputParser()
    | RunnableLambda(create_summary_prompt)
    | model
    | StrOutputParser()
)
res = explanation_chain.invoke({"topic": "파이썬"})
print(f' ==> [Line 19]: \033[38;2;161;242;21m[res]\033[0m({type(res).__name__}) = \033[38;2;231;90;110m{res}\033[0m')

# 체인 흐름: {topic} → 설명 → 문자열 요약 프롬프트 → LLM → 문자열
chain = (
    explanation_prompt
    | model 
    | StrOutputParser()
    | RunnableLambda(create_summary_prompt)  # 문자열 → 프롬프트 문자열 변환
    | model
    | StrOutputParser()
)

result = chain.invoke({"topic": "파이썬"})
print("최종 요약 결과:")
print(result)

 ==> [Line 19]: [res](str) = 파이썬은 1991년 귀도 반 로썸에 의해 개발된 범용 프로그래밍 언어로, 가독성이 높고 다양한 라이브러리와 플랫폼 독립성을 제공하여 초보자가 배우기 쉽고 다양한 분야에서 널리 활용됩니다.
최종 요약 결과:
파이썬은 1991년에 개발된 간결하고 읽기 쉬운 고수준 프로그래밍 언어로, 다양한 라이브러리와 플랫폼 지원, 쉬운 디버깅으로 인해 초보자와 전문가 모두에게 인기가 높습니다.


## 6. OutputParser를 활용한 구조화된 출력

다단계 체인을 다뤄봤어요. 이번에는 LLM 응답을 딕셔너리 형태로 파싱하는 커스텀 파서를 만들어볼게요.

`StrOutputParser`는 단순 문자열만 반환해요. 더 복잡한 구조가 필요할 때는 `RunnableLambda`로 파싱 함수를 체인에 연결할 수 있어요.

> **실무 팁**: 프로덕션 환경에서는 LLM 출력 형식이 일정하지 않을 수 있어요. `PydanticOutputParser`나 `JsonOutputParser` 같은 견고한 파서를 사용하는 편이 안정적이에요. 이 내용은 이후 노트북에서 다룰 거예요.

In [ ]:
# ---------------------------------------------------
# RunnableLambda로 LLM 응답을 구조화된 딕셔너리로 파싱하기
# ---------------------------------------------------


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **`PromptTemplate`**: `{변수명}` 플레이스홀더로 재사용 가능한 프롬프트를 만들어요
- **`StrOutputParser`**: `AIMessage`에서 텍스트 문자열만 추출해요
- **`RunnableLambda`**: 일반 Python 함수를 파이프에 연결할 수 있는 Runnable(러너블)로 변환해요
- **파이프 연산자(`|`)**: 여러 Runnable을 연결해 데이터 흐름을 직관적으로 표현해요
- **다단계 체인**: `RunnableLambda`로 체인 간 데이터 형식을 변환해 여러 LLM 호출을 연결할 수 있어요

## 다음 노트북 예고

다음 `03_LCEL_basic.ipynb`에서는 LangChain Expression Language(LCEL)의 표준 인터페이스인 `invoke`, `batch`, `stream`과 비동기 메서드를 더 깊이 다뤄요. 체인을 다양한 방식으로 실행하는 법을 알아볼게요.